# Explorando os dados — BAR-ATH (tactical)

Visão geral de tudo o que está disponível em `data/` e nos ficheiros pré-processados:

1. Ficheiros disponíveis
2. Metadados do ATD (equipas e *track slots*)
3. Tracking ATD — formato bruto (com velocidade) e matriz em metros
4. Presença da bola e nº de jogadores por frame
5. Um frame no campo + mapa de calor da bola
6. Homografia (tactical e broadcast)
7. Smart Tagging
8. Ground truth público
9. Features pré-calculadas (`viewer/features.py` + `features_coletivas.py`)

In [ ]:
%matplotlib inline
import os, sys, json, warnings
import xml.etree.ElementTree as ET
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)

HERE = os.path.abspath("") if os.path.exists("rotulos.py") else os.path.join(os.path.abspath(""), "analise")
ROOT = os.path.dirname(HERE)
for p in (ROOT, os.path.join(ROOT, "viewer"), os.path.join(ROOT, "starter"), HERE):
    if p not in sys.path: sys.path.insert(0, p)

from games import DATA_DIR, GAME, GAMES, asset_paths, half_video_blocks
FPS = GAMES[GAME]["fps"]; LENGTH, WIDTH = GAMES[GAME]["pitch"]
PT, PB = asset_paths(), asset_paths(source="broadcast")
HALVES = half_video_blocks(GAME)
COL = {"H": "#a50044", "A": "#1f77b4", "D": "#111111", "X": "#cccccc"}
minute = lambda fr: np.asarray(fr) / FPS / 60
sns.set_theme(style="whitegrid", context="notebook")
print("DATA_DIR:", DATA_DIR); print("metades avaliadas:", HALVES)

## 1. Ficheiros disponíveis

In [ ]:
rows = []
for base in (DATA_DIR, os.path.join(ROOT, "viewer", "data"), os.path.join(ROOT, "evaluation", "gt")):
    for dirpath, _, files in os.walk(base):
        for fn in files:
            fp = os.path.join(dirpath, fn)
            rows.append(dict(pasta=os.path.relpath(dirpath, ROOT), ficheiro=fn,
                             MB=round(os.path.getsize(fp) / 1e6, 2)))
files = pd.DataFrame(rows).sort_values(["pasta", "ficheiro"]).reset_index(drop=True)
files

## 2. Metadados do ATD (XML)
Equipas e a equipa de cada um dos 46 *track slots* (um slot **não** é um jogador).

In [ ]:
root = ET.parse(PT["atd_meta"]).getroot()
teams = pd.DataFrame([dict(id=t.get("id"), nome=t.findtext("Name")) for t in root.iter("Team")])
display(teams)
slots = pd.DataFrame([dict(slot_id=p.get("id"), teamId=p.get("teamId"), nome=p.findtext("Name"),
                           shirt=p.findtext("ShirtNumber")) for p in root.iter("Player")])
slots["lado"] = np.where(slots.teamId == teams.id[0], "H", "A")
print(slots.lado.value_counts().to_dict())
slots.head(10)

## 3. Tracking ATD

### 3a. Formato bruto (`_atd.txt`)
Uma linha por frame: `frame_id:jog1_x,jog1_y,jog1_v;...;jog46_x,jog46_y,jog46_v:bola_x,bola_y,bola_v`.
Coordenadas normalizadas 0–1, velocidade em m/s. Lemos só as primeiras linhas a partir do início do 1º tempo.

In [ ]:
def read_raw(path, start_frame_id, n):
    recs = []
    with open(path) as fh:
        for line in fh:
            fid, pl, ball = line.rstrip("\n").split(":")
            fid = int(fid)
            if fid < start_frame_id: continue
            r = {"frame_id": fid}
            for k, g in enumerate(pl.split(";"), 1):
                x, y, v = g.split(",")
                r[f"{k}_x"], r[f"{k}_y"], r[f"{k}_s"] = float(x), float(y), float(v)
            bx, by, bv = ball.split(",")
            r["ball_x"], r["ball_y"], r["ball_s"] = float(bx), float(by), float(bv)
            recs.append(r)
            if len(recs) >= n: break
    return pd.DataFrame(recs).set_index("frame_id")

raw = read_raw(PT["atd_raw"], start_frame_id=HALVES[0][1] + 1, n=2000)   # frame_id = video_frame + 1
print(raw.shape)
raw[["ball_x", "ball_y", "ball_s"] + [c for c in raw.columns if c.startswith(("1_", "25_"))]].head()

### 3b. Matriz completa em metros (`viewer/data/atd_tactical.bin`)
`players` é (frames, 46, 2), `ball` é (frames, 2); índice = frame do vídeo.

In [ ]:
from features import load_atd
players, ball, slot_teams, meta = load_atd()
N = len(ball)
print("players", players.shape, " ball", ball.shape, " pitch", meta["pitch"])

graded = np.zeros(N, bool)
for _, s, e in HALVES: graded[s:e + 1] = True
ball_ok = ~np.isnan(ball[:, 0])
n_vis = (~np.isnan(players[..., 0])).sum(1)
n_H = (~np.isnan(players[:, slot_teams == "H", 0])).sum(1)
n_A = (~np.isnan(players[:, slot_teams == "A", 0])).sum(1)

pd.DataFrame({
    "todos os frames": [N, ball_ok.mean(), n_vis.mean(), n_H.mean(), n_A.mean()],
    "frames avaliados": [graded.sum(), ball_ok[graded].mean(), n_vis[graded].mean(),
                         n_H[graded].mean(), n_A[graded].mean()],
}, index=["n frames", "bola visível (%)", "objetos/frame", "slots H/frame", "slots A/frame"]).round(3)

In [ ]:
# Tabela "longa" de um trecho: uma linha por (frame, slot)
fr0 = HALVES[0][1]
sl = slice(fr0, fr0 + 250)
long = pd.DataFrame({
    "frame": np.repeat(np.arange(N)[sl], players.shape[1]),
    "slot": np.tile(np.arange(1, players.shape[1] + 1), sl.stop - sl.start),
    "lado": np.tile(slot_teams, sl.stop - sl.start),
    "x_m": players[sl, :, 0].ravel(), "y_m": players[sl, :, 1].ravel(),
}).dropna()
long.head(12)

## 4. Presença da bola e nº de objetos ao longo do jogo

In [ ]:
w = 25 * 30   # janela de 30 s
fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)
axes[0].plot(minute(np.arange(N)), pd.Series(ball_ok.astype(float)).rolling(w, center=True).mean(), lw=1)
axes[0].set_ylabel("bola visível\n(média 30 s)")
axes[1].plot(minute(np.arange(N)), pd.Series(n_vis).rolling(w, center=True).mean(), lw=1, color="k")
axes[1].axhline(22, ls="--", c="gray"); axes[1].set_ylabel("objetos/frame")
axes[2].plot(minute(np.arange(N)), pd.Series(n_H).rolling(w, center=True).mean(), c=COL["H"], label="H")
axes[2].plot(minute(np.arange(N)), pd.Series(n_A).rolling(w, center=True).mean(), c=COL["A"], label="A")
axes[2].legend(); axes[2].set_ylabel("slots por lado"); axes[2].set_xlabel("minuto do vídeo")
for ax in axes:
    for _, s, e in HALVES: ax.axvspan(minute(s), minute(e), color="green", alpha=0.06)
plt.tight_layout(); plt.show()

# Duração dos buracos sem bola (frames avaliados)
from itertools import groupby
gaps = [len(list(g)) for k, g in groupby(~ball_ok[graded]) if k]
gaps = pd.Series(gaps) / FPS
print("buracos sem bola: n={}, mediana {:.1f}s, p90 {:.1f}s, máx {:.1f}s".format(
    len(gaps), gaps.median(), gaps.quantile(.9), gaps.max()))

In [ ]:
# Ocupação dos slots: quantos frames cada slot está ativo
occ = pd.DataFrame({"slot": np.arange(1, players.shape[1] + 1), "lado": slot_teams,
                    "frames_ativos": (~np.isnan(players[..., 0])).sum(0)})
fig, ax = plt.subplots(figsize=(15, 3))
ax.bar(occ.slot, occ.frames_ativos / FPS / 60, color=[COL[t] for t in occ.lado])
ax.set_xlabel("slot"); ax.set_ylabel("minutos ativo"); ax.set_title("ocupação dos track slots")
plt.show()

## 5. Um frame no campo e mapa de calor da bola

In [ ]:
from utils import draw_pitch

def show_frame(fr, ax=None):
    ax = draw_pitch(ax, LENGTH, WIDTH)
    for t in "HA":
        p = players[fr, slot_teams == t]
        ax.scatter(p[:, 0], p[:, 1], s=80, c=COL[t], edgecolor="w", zorder=3, label=meta["team_names"][t])
    if ball_ok[fr]:
        ax.scatter(*ball[fr], s=60, c="w", edgecolor="k", zorder=4, label="bola")
    ax.set_title("frame {}  ({:.2f} min){}".format(fr, fr / FPS / 60, "" if ball_ok[fr] else "  — sem bola"))
    ax.legend(loc="upper right", fontsize=8)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
show_frame(HALVES[0][1] + 25 * 60, axes[0])
show_frame(HALVES[1][1] + 25 * 600, axes[1])
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, (half, s, e) in zip(axes, HALVES):
    b = ball[s:e + 1]; b = b[~np.isnan(b[:, 0])]
    draw_pitch(ax, LENGTH, WIDTH)
    ax.hexbin(b[:, 0], b[:, 1], gridsize=(35, 18), extent=(0, LENGTH, 0, WIDTH), cmap="magma", alpha=.8, mincnt=1)
    ax.set_title(f"posição da bola — {half}º tempo")
plt.show()

## 6. Homografia
Uma matriz 3×3 por frame (campo 0–1 → píxeis). `valid` diz se existe; `reprojection_error` quanto confiar.

In [ ]:
homT = pd.read_csv(PT["homography"])
homB = pd.read_csv(PB["homography"])
display(homT.head(3))
pd.DataFrame({
    "tactical": [len(homT), homT.valid.mean(), homT.reprojection_error.median(), homT.reprojection_error.max()],
    "broadcast": [len(homB), homB.valid.mean(), homB.reprojection_error.median(), homB.reprojection_error.max()],
}, index=["frames", "válida (%)", "erro mediano", "erro máx"]).round(3)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 5))
axes[0].plot(minute(homT.frame), homT.reprojection_error.rolling(250).median(), lw=1)
axes[0].set_ylabel("erro reproj.\n(tactical)"); axes[0].set_yscale("log")
axes[1].plot(minute(homB.frame), homB.valid.rolling(250).mean(), lw=1, c="tab:orange")
axes[1].set_ylabel("válida\n(broadcast)"); axes[1].set_xlabel("minuto do vídeo (cada um na sua timeline)")
plt.tight_layout(); plt.show()

## 7. Smart Tagging
638 fases automáticas (não é ground truth). `start_frame = round(start_s * 25)`.

In [ ]:
import load_smart_tagging as LST
ev = pd.DataFrame(LST.load())
tags = pd.json_normalize(ev.pop("tags"))
st = pd.concat([ev, tags], axis=1)
st["dur_s"] = st.end_s - st.start_s
print(st.shape); display(st.head())
display(st.code.value_counts().to_frame("n"))

In [ ]:
for col in ["Team", "Half", "Type", "Side", "Direction of ball entry", "Max Players in the box"]:
    if col in st: print(col, st[col].value_counts(dropna=False).to_dict())
display(st.groupby("code").dur_s.describe().round(1))

In [ ]:
codes = st.code.value_counts().index.tolist()
fig, ax = plt.subplots(figsize=(15, 6))
tc = {"Barcelona": COL["H"], "Atletic Club": COL["A"], "N/A": "gray"}
for i, c in enumerate(codes):
    sub = st[st.code == c]
    ax.broken_barh(list(zip(minute(sub.start_frame), minute(sub.end_frame - sub.start_frame))), (i - .4, .8),
                   facecolors=[tc.get(t, "gray") for t in sub.Team])
ax.set_yticks(range(len(codes))); ax.set_yticklabels(codes, fontsize=8); ax.invert_yaxis()
for _, s, e in HALVES: ax.axvspan(minute(s), minute(e), color="green", alpha=0.05)
ax.set_xlabel("minuto do vídeo"); ax.set_title("Smart Tagging (vermelho = Barcelona, azul = Athletic, cinza = N/A)")
plt.show()

## 8. Ground truth público
Rótulos reais (`H`/`A`/`D`) só nos primeiros 5 minutos de jogo (frames 11185–18684); o resto é `X`.

In [ ]:
gt = pd.read_csv(os.path.join(ROOT, "evaluation", "gt", f"{GAME}_possession_gt_public.csv"))
print(gt.label.value_counts().to_dict())
g = gt[gt.label != "X"]
runs = (g.label != g.label.shift()).cumsum()
segs = g.groupby(runs).agg(label=("label", "first"), inicio=("frame", "first"), fim=("frame", "last"))
segs["dur_s"] = (segs.fim - segs.inicio + 1) / FPS
display(segs.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(15, 1.8))
for _, r in segs.iterrows():
    ax.axvspan(minute(r.inicio), minute(r.fim + 1), color=COL[r.label], lw=0)
ax.set_yticks([]); ax.set_xlim(minute(g.frame.min()), minute(g.frame.max())); ax.grid(False)
ax.set_xlabel("minuto do vídeo"); ax.set_title("GT público (vermelho H, azul A, preto D)")
plt.show()

## 9. Features pré-calculadas
Matriz (frames × features) de `viewer/features.py` + `analise/features_coletivas.py`.

In [ ]:
from features_coletivas import load_all
X, fmeta = load_all()
I = fmeta["index"]
print("X:", X.shape)
cat = pd.DataFrame(fmeta["features"])
cat["missing_avaliados"] = np.isnan(X[graded]).mean(0).round(3)
cols = [c for c in ["name", "unit", "description", "missing_avaliados", "p05", "p50", "p95"] if c in cat]
cat[cols]

In [ ]:
# DataFrame com as features nos frames do GT público, para explorar à vontade
cov = (gt.label != "X").values
F = pd.DataFrame(X[cov], columns=[f["name"] for f in fmeta["features"]], index=gt.frame[cov])
F["gt"] = gt.label[cov].values
F.groupby("gt").median().T.head(40)

In [ ]:
# Escolhe uma feature e compara a distribuição por classe do GT
feat = "ball_speed" if "ball_speed" in F else F.columns[0]
fig, ax = plt.subplots(figsize=(8, 4))
for lab in "HAD":
    sns.kdeplot(F.loc[F["gt"] == lab, feat].dropna(), ax=ax, label=lab, color=COL[lab], fill=True, alpha=.2)
ax.set_title(feat); ax.legend(); plt.show()